<a href="https://colab.research.google.com/github/DanielTrindade/nota-secreta-a2a/blob/main/nota_secreta_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Nota Secreta — A2A / LLM (Colab)

Notebook pronto para rodar o jogo **Nota Secreta** no Google Colab.

O projeto sobe, como subprocessos HTTP:

- um **serviço LLM** centralizado (`llm_service.py`);
- o **Game Master** (`game_master.py`);
- o **nosso agente estratégico** + agentes `random_agent.py` (ou um campo todo estratégico).

Temos duas versões do nosso agente:

- `llm_agent.py` — versão base;
- `llm_agent_v2.py` — versão **calibrada** (dica do narrador na "zona Dixit" + blefe otimizado). É a que usamos por padrão aqui, via `--agent llm_agent_v2.py`.

> **Por que rodamos via `!python ...` e não via `import`?**
> O `run_game.py`/`ab_test.py` usam `asyncio.run(...)`, que conflita com o event loop já ativo do Jupyter/Colab. Rodar como processo de shell evita esse conflito e mantém o código original intacto.

**Fluxo recomendado:** seções 1 → 2 → 3 → 4 (jogo em mock, valida a infra) e o smoke do A/B (4.3). Depois a **seção 5 (modelo real)** para jogar de verdade e rodar o **teste A/B que vale (5.4)** — é só lá que a calibração do narrador age.

## 1. Clonar o repositório

O `REPO_URL` já aponta para o repositório. Se quiser usar um fork, troque a URL abaixo.

In [1]:
REPO_URL = "https://github.com/DanielTrindade/nota-secreta-a2a.git"

import os
repo_dir = REPO_URL.rstrip('/').split('/')[-1].replace('.git', '')
if not os.path.isdir(repo_dir):
    !git clone $REPO_URL
%cd $repo_dir
!ls -la

Cloning into 'nota-secreta-a2a'...
remote: Enumerating objects: 21, done.
remote: Counting objects: 100% (21/21), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 21 (delta 3), reused 21 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (21/21), 117.43 KiB | 985.00 KiB/s, done.
Resolving deltas: 100% (3/3), done.
/content/nota-secreta-a2a
total 404
drwxr-xr-x 4 root root   4096 Jun  8 15:37 .
drwxr-xr-x 1 root root   4096 Jun  8 15:37 ..
-rw-r--r-- 1 root root  12753 Jun  8 15:37 base_agent.py
-rw-r--r-- 1 root root 263339 Jun  8 15:37 brazilian_songs.csv
-rw-r--r-- 1 root root   4834 Jun  8 15:37 fasta2a.py
-rw-r--r-- 1 root root  23224 Jun  8 15:37 game_master.py
drwxr-xr-x 8 root root   4096 Jun  8 15:37 .git
-rw-r--r-- 1 root root    124 Jun  8 15:37 .gitignore
-rw-r--r-- 1 root root  19538 Jun  8 15:37 llm_agent.py
-rw-r--r-- 1 root root   4642 Jun  8 15:37 llm_service.py
-rw-r--r-- 1 root root   6424 Jun  8 15:37 nota_secreta_colab.ipynb
-rw-r--r-- 1 r

## 2. Instalar dependências (modo mock)

Para validar a arquitetura em modo mock **não** é preciso instalar `llama-cpp-python` (que é pesado e compila). Instalamos só o essencial:

In [2]:
!pip install -q "fastapi>=0.110" "uvicorn>=0.29" "pydantic>=2.6" "aiohttp>=3.9" "pytest>=8.0"

## 3. Rodar os testes

Roda a suíte completa: a pontuação do jogo (`tests/test_scoring.py`) e o nosso agente calibrado (`tests/test_llm_agent_v2.py` — classificador de dificuldade da dica, desempate do blefe e as 5 tools via *monkeypatch* da LLM).

In [ ]:
!python -m pytest tests -v

## 4. Partida em modo mock (com o nosso agente)

Roda uma partida completa **sem modelo real** (rápido, ótimo para validar a infraestrutura e o protocolo). O serviço LLM responde com um texto fixo, então o agente cai nos seus *fallbacks* — útil para testar robustez, não a qualidade semântica.

Usamos o nosso agente calibrado via `--agent llm_agent_v2.py`. (Para a versão base, troque por `llm_agent.py` ou omita o `--agent`.)

Ao final, o caminho do log é mostrado no terminal e salvo em `logs/`.

In [ ]:
!python run_game.py --agent llm_agent_v2.py --force-mock

Variante: **6 agentes estratégicos** em modo mock (todos usando o nosso `llm_agent_v2.py`):

In [ ]:
!python run_game.py --agent llm_agent_v2.py --all-strategic --force-mock

## 4.3. Smoke test do A/B (mock, rápido)

`ab_test.py` compara `llm_agent.py` (base) com `llm_agent_v2.py` (calibrado) **na mesma partida**, alternando as cadeiras a cada jogo para anular a vantagem do narrador inicial.

> ⚠️ **Esta célula roda em mock só para validar a mecânica do harness.** Em mock a LLM devolve sempre o mesmo texto e a calibração do narrador fica **inerte** — os dois agentes tendem a empatar. **A comparação que vale está na seção 5.4** (modelo real), depois de baixar o GGUF.

In [ ]:
# Smoke test (mock): valida o harness; NÃO mede qualidade semântica. Veja 5.4 para o teste real.
!python ab_test.py --games 4 --force-mock --target-score 15

## 5. (Opcional) Partida com modelo real (GGUF)

Esta seção usa um modelo LLM real (`Phi-3.5-mini-instruct`, ~2,4 GB) via `llama-cpp-python`.

**Atenção:**
- A instalação do `llama-cpp-python` **compila** e pode levar alguns minutos.
- O download do modelo (~2,4 GB) também demora.
- Para acelerar a inferência, use um runtime com **GPU** no Colab (`Ambiente de execução → Alterar tipo de ambiente de execução → GPU`).

### 5.1. Instalar o `llama-cpp-python`

In [5]:
# CPU (sempre funciona). Para GPU, veja a célula alternativa abaixo.
!pip install -q "llama-cpp-python>=0.2.70"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
# (Alternativa GPU) Instala wheel com suporte a CUDA — use só se o runtime tiver GPU.
# !CMAKE_ARGS="-DGGML_CUDA=on" pip install -q --force-reinstall --no-cache-dir "llama-cpp-python>=0.2.70"

### 5.2. Baixar o modelo GGUF

In [4]:
import os
os.makedirs('models', exist_ok=True)
MODEL_PATH = 'models/phi-3.5-mini-instruct-q4_k_m.gguf'
MODEL_URL = 'https://huggingface.co/bartowski/Phi-3.5-mini-instruct-GGUF/resolve/main/Phi-3.5-mini-instruct-Q4_K_M.gguf'
if not os.path.exists(MODEL_PATH):
    !wget -q --show-progress -O $MODEL_PATH $MODEL_URL
!ls -lh models/

models/phi-3.5-mini 100%[===================>]   2.23G   220MB/s    in 13s     
total 2.3G
-rw-r--r-- 1 root root 2.3G Jun  8 15:43 phi-3.5-mini-instruct-q4_k_m.gguf


### 5.3. Rodar a partida com o modelo real (nosso agente)

`--llm-max-concurrency 1` evita sobrecarregar o modelo local (uma requisição por vez). Usamos `--agent llm_agent_v2.py` para jogar com o nosso agente calibrado — é aqui que a calibração da dica e o blefe realmente fazem efeito (a LLM gera texto de verdade).

In [ ]:
!python run_game.py --agent llm_agent_v2.py --model models/phi-3.5-mini-instruct-q4_k_m.gguf --llm-max-concurrency 1

### 5.4. Teste A/B com o modelo real — a comparação que vale

Com o modelo carregado, este é o A/B **de verdade**: é aqui que a calibração do narrador age (a LLM gera dicas reais e a banda decide quando regerar). Compara `llm_agent.py` vs `llm_agent_v2.py` em **campo todo estratégico** (3 vs 3), alternando as cadeiras a cada partida.

Além de pontos totais e vitórias, o relatório mostra o desempenho **como narrador** — pontos e taxa de acerto da "zona Dixit" (alguns mas não todos acertam) — que é exatamente o que a calibração tenta melhorar.

> ⏱️ Em CPU é lento (muitas chamadas à LLM em série). Comece com poucas partidas; para um primeiro resultado mais rápido, baixe `--target-score` (ex.: 15). Aumente `--games` para um veredito mais estável.
>
> 🔒 O `ab_test.py` aborta se o `llama-cpp` não carregar (cairia em mock sem você perceber) — então, se ele rodar até o fim, o teste foi mesmo com o modelo real.

In [ ]:
# A/B REAL: 3 base vs 3 calibrado, com o modelo do professor (3 vs 3, cadeiras alternadas).
# --model ja aponta para o GGUF padrao; baixe --target-score (ex.: 15) para um 1o resultado mais rapido.
!python ab_test.py --all-strategic --games 4 --target-score 30 --model models/phi-3.5-mini-instruct-q4_k_m.gguf

## 6. Ler o log da última partida

Transforma o log JSON mais recente numa visualização legível (narrador, dicas, cartas jogadas, votos e evolução da pontuação).

In [7]:
import glob, os
logs = sorted(glob.glob('logs/*.json'), key=os.path.getmtime)
if logs:
    ultimo = logs[-1]
    print('Log mais recente:', ultimo)
    !python render_log_readable.py "$ultimo"
else:
    print('Nenhum log encontrado — rode uma partida nas seções 4 ou 5 primeiro.')

Log mais recente: logs/partida_20260608_161009.json
Log: 2026-06-08T16:10:09.980255

Letras
id  type       name           card_id  title                         lyrics                                                                          
--  ---------  -------------  -------  ----------------------------  --------------------------------------------------------------------------------
0   strategic  LLMAgent_1     172      Sá Marina                     Descendo a rua da ladeira Só quem viu, que pode contar Cheirando a flor de lara…
                              143      Minha Namorada                Se você quer ser minha namorada Ah, que linda namorada Você poderia ser Se quis…
                              67       O Canto da Cidade             A cor dessa cidade sou eu O canto dessa cidade é meu A cor dessa cidade sou eu… 
                              201      Fixação                       Seu rosto na tevê parece um milagre Uma perfeição nos mínimos detalhes Eu mudo… 
1   rand